# BƯỚC 1: KHÁM PHÁ VÀ PHÂN TÍCH DỮ LIỆU (EXPLORATORY DATA ANALYSIS - EDA)

## Mục tiêu theo đề bài Practice Exercise 3:
1. Đọc và kiểm tra cấu trúc tập dữ liệu khách hàng (`data/raw/marketing_campaign.csv`, định dạng phân tách bằng tab `\t`).
2. Tạo các đặc trưng phân cụm cốt lõi (`Age = 2014 - Year_Birth`, `Total_Spending = sum(6 cột Mnt*)`), lọc lỗi miền bất hợp lý (`Year_Birth < 1920`, `Income > 200,000`) và chia 80% Train / 20% Test trước khi điền trung vị `Income`.
3. Phân tích giá trị khuyết thiếu (Missing values), phát hiện dị biệt (Outliers) bằng phương pháp IQR (Tukey's Fences).
4. Khảo sát hình thái phân phối, quan hệ song biến và biến nhân khẩu học thật (`Education`, `Marital_Status`).

In [ ]:
# Thêm đường dẫn gốc của dự án để import module từ src/
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    NOTEBOOK_FIGURES_DIR,
    NOTEBOOK_METRICS_DIR,
    NOTEBOOK_PROCESSED_DIR,
    NUMERICAL_FEATURES,
    RAW_DATA_PATH,
)
from src.data.loader import load_raw_data, split_data, validate_schema
from src.data.preprocessing import (
    add_derived_features,
    audit_missing_and_outliers,
    clean_data,
    compute_train_medians,
    filter_invalid_records,
)
from src.evaluation.visualization import plot_eda_correlations, plot_eda_distributions

NOTEBOOK_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
NOTEBOOK_METRICS_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
# Nạp dữ liệu gốc và kiểm định tính toàn vẹn schema
raw_df = load_raw_data(RAW_DATA_PATH)
schema_info = validate_schema(raw_df, allow_duplicate_ids=True)
if not schema_info["is_valid"]:
    raise ValueError(f"Schema không hợp lệ: {schema_info}")
raw_df.head()

 [NẠP DỮ LIỆU] Đã đọc thành công 2,240 dòng x 29 cột từ marketing_campaign.csv
 [KIỂM ĐỊNH SCHEMA] Trạng thái: HỢP LỆ | Dòng: 2,240 | Cột thiếu: 0 | ID trùng: 0 | ID thiếu: 0 | Giá trị khuyết: 24


,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,...,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response
0,5524,1957,Graduation,Single,58138.0,0,0,04-09-2012,58,635,...,7,0,0,0,0,0,0,3,11,1
1,2174,1954,Graduation,Single,46344.0,1,1,08-03-2014,38,11,...,5,0,0,0,0,0,0,3,11,0
2,4141,1965,Graduation,Together,71613.0,0,0,21-08-2013,26,426,...,4,0,0,0,0,0,0,3,11,0
3,6182,1984,Graduation,Together,26646.0,1,0,10-02-2014,26,11,...,6,0,0,0,0,0,0,3,11,0
4,5324,1981,PhD,Married,58293.0,1,0,19-01-2014,94,173,...,5,0,0,0,0,0,0,3,11,0


In [ ]:
# Kiểm toán giá trị khuyết, ngoại lai (IQR) và dị biệt hạng mục trên dữ liệu gốc trước khi làm sạch
raw_audit_df = audit_missing_and_outliers(raw_df)
raw_audit_df.to_csv(NOTEBOOK_METRICS_DIR / "raw_data_audit.csv", index=False)
print("Giá trị Marital_Status thô:", raw_df["Marital_Status"].value_counts().to_dict())
print("Năm sinh < 1920:", raw_df[raw_df["Year_Birth"] < 1920][["ID", "Year_Birth", "Income"]].to_dict(orient="records"))
print("Thu nhập > 200,000:", raw_df[raw_df["Income"] > 200000][["ID", "Year_Birth", "Income"]].to_dict(orient="records"))
raw_audit_df

In [ ]:
# Lọc bản ghi bất hợp lý, chia 80% Train / 20% Test rồi mới điền trung vị Income của tập Train
valid_df = add_derived_features(filter_invalid_records(raw_df))
train_raw, test_raw = split_data(valid_df, save_files=False)
train_medians = compute_train_medians(train_raw)

train_df = clean_data(train_raw, fill_medians=train_medians, drop_invalid_outliers=False)
test_df = clean_data(test_raw, fill_medians=train_medians, drop_invalid_outliers=False)

NOTEBOOK_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
train_df.to_csv(NOTEBOOK_PROCESSED_DIR / "train.csv", index=False)
test_df.to_csv(NOTEBOOK_PROCESSED_DIR / "test.csv", index=False)
pd.concat([train_df, test_df], ignore_index=True).to_csv(
    NOTEBOOK_PROCESSED_DIR / "full_customers.csv", index=False
)

train_audit_df = audit_missing_and_outliers(train_df)
train_audit_df.to_csv(NOTEBOOK_METRICS_DIR / "train_iqr_audit.csv", index=False)

print("Thống kê mô tả trên tập Train (80%):")
train_df[NUMERICAL_FEATURES].describe().T

In [ ]:
# Trực quan hóa phân phối, Boxplot phát hiện ngoại lai và tương quan song biến trên tập Train (lưu riêng vào notebooks/results/figures)
plot_eda_distributions(train_df, output_dir=NOTEBOOK_FIGURES_DIR)
plot_eda_correlations(train_df, output_dir=NOTEBOOK_FIGURES_DIR)